# C12-classical-models — Session 6: Lloyd's k-means and Classical-Model Comparison

*One 90-minute session. Cluster indices are `0, ..., k-1`; exact distance ties choose the smaller
cluster index.*


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-8
rng = np.random.default_rng(SEED)


## 1. Unsupervised objective and geometry

k-means receives features $X\in\mathbb R^{N\times D}$ without target labels. For assignments
$c_i\in\{0,\ldots,k-1\}$ and centroids $\mu_j\in\mathbb R^D$, it minimizes within-cluster sum
of squares (WCSS)

$$J(c,\mu)=\sum_{i=1}^N\|x_i-\mu_{c_i}\|_2^2.$$

scikit-learn calls the fitted WCSS `inertia_`. Euclidean distance yields Voronoi cells separated
by linear bisectors, favoring compact roughly spherical groups. Cluster numbers have no semantic
ordering.

**Checkpoint 1A.** Is WCSS a supervised prediction error?

**Checkpoint 1B.** What fitted attribute stores WCSS in `KMeans`?


## 2. Lloyd assignment and centroid update

Lloyd's algorithm alternates:

1. **Assignment:** $c_i\leftarrow\arg\min_j\|x_i-\mu_j\|^2$, with the smaller index on an exact
   tie (`np.argmin` already chooses the first minimum).
2. **Update:** $\mu_j\leftarrow$ arithmetic mean of rows assigned to cluster $j$.

Stop when assignments do not change or the maximum centroid movement is at most a declared
`tol`; also cap iterations. The output contract includes labels `(N,)` integer, centroids `(k,D)`
float, and an objective trace.

**Checkpoint 2A.** Which NumPy behavior implements the tie rule?

**Checkpoint 2B.** What is the centroid of points `(0,2)` and `(2,4)`?


In [ ]:
def assign_points(X, centroids):
    squared = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return np.argmin(squared, axis=1).astype(np.int64)

X_small = np.array([[0., 0.], [0., 2.], [4., 0.], [4., 2.]])
mu0 = np.array([[0., 0.], [4., 0.]])
labels0 = assign_points(X_small, mu0)
mu1 = np.vstack([X_small[labels0 == j].mean(axis=0) for j in range(2)])
assert np.array_equal(labels0, np.array([0, 0, 1, 1]))
assert np.allclose(mu1, np.array([[0., 1.], [4., 1.]]), atol=ATOL, rtol=RTOL)
print(labels0, mu1)


## 3. Why each step cannot increase WCSS

With centroids fixed, assignment chooses the nearest centroid separately for each row, so no
other assignment can lower that row's contribution. With assignments fixed, for one nonempty
cluster the identity

$$\sum_{i\in C}\|x_i-\mu\|^2=\sum_{i\in C}\|x_i-\bar x_C\|^2+|C|\|\mu-\bar x_C\|^2$$

shows the arithmetic mean minimizes squared distance. Therefore assignment then update cannot
increase WCSS. The algorithm terminates at a local fixed point, not necessarily the global
minimum.

**Checkpoint 3A.** Which nonnegative term proves the mean is optimal?

**Checkpoint 3B.** Does monotone WCSS prove all initializations reach the same solution?


## 4. Initialization, `k-means++`, and empty clusters

Initialization matters because the objective is nonconvex jointly. `k-means++` selects spread-out
seeds probabilistically, and `n_init` repeats full fits; choose the lowest final inertia.
For a from-scratch deterministic implementation here, initial indices are supplied explicitly.

An update can encounter an empty cluster. This unit's robust policy is: preserve the squared
residuals from the assignment step; for the smallest-index empty cluster, reassign the eligible
row with the largest preserved residual, breaking ties by smaller row index. A row is eligible
only if its current donor cluster has at least two members. Recompute membership counts after each
move, repeat until none are empty, and then recompute all means.
This rule is part of the algorithm, not an implementation detail.

**Checkpoint 4A.** What does `n_init` protect against?

**Checkpoint 4B.** Under this unit's policy, which row fills an empty cluster?


## 5. Worked Lloyd implementation and objective trace

`lloyd` below validates `X` shape `(N,D)`, initial centroids `(k,D)`, finite values, and
$1\le k\le N$. Each recorded objective is computed after the empty-cluster repair and centroid
update using the current labels/centroids. The trace must be nonincreasing up to `atol=1e-10,
rtol=1e-8`.

**Checkpoint 5A.** Why must an objective be computed from a mutually consistent label/centroid
pair?

**Checkpoint 5B.** What two stopping safeguards prevent an endless loop?


In [ ]:
def lloyd(X, initial_centroids, max_iter=100, tol=1e-10):
    X = np.asarray(X, dtype=np.float64)
    centroids = np.asarray(initial_centroids, dtype=np.float64).copy()
    if X.ndim != 2 or centroids.ndim != 2 or X.shape[1] != centroids.shape[1]:
        raise ValueError("incompatible 2D shapes")
    if not np.isfinite(X).all() or not np.isfinite(centroids).all():
        raise ValueError("inputs must be finite")
    k = centroids.shape[0]
    if not (1 <= k <= X.shape[0]):
        raise ValueError("require 1 <= k <= N")
    trace = []
    previous = None
    for _ in range(max_iter):
        labels = assign_points(X, centroids)
        distances = ((X - centroids[labels]) ** 2).sum(axis=1)
        for empty in np.flatnonzero(np.bincount(labels, minlength=k) == 0):
            available = np.flatnonzero(np.bincount(labels, minlength=k)[labels] > 1)
            farthest = available[np.argmax(distances[available])]
            labels[farthest] = empty
        updated = np.vstack([X[labels == j].mean(axis=0) for j in range(k)])
        objective = float(((X - updated[labels]) ** 2).sum())
        trace.append(objective)
        movement = float(np.max(np.linalg.norm(updated - centroids, axis=1)))
        unchanged = previous is not None and np.array_equal(labels, previous)
        centroids, previous = updated, labels.copy()
        if unchanged or movement <= tol:
            break
    return labels, centroids, np.asarray(trace, dtype=np.float64)

X_fit = np.array([[-2., 0.], [-1.5, 0.5], [-1., -0.5],
                  [2., 0.], [1.5, -0.5], [1., 0.5]])
labels, centers, trace = lloyd(X_fit, X_fit[[0, 3]])
assert labels.shape == (6,) and labels.dtype == np.int64
assert centers.shape == (2, 2) and centers.dtype == np.float64
assert np.all(np.diff(trace) <= ATOL)
print("labels", labels, "centers", centers, "trace", trace)


## 6. Scaling, diagnostics, and sklearn comparison

Because WCSS uses Euclidean distances, standardize features when their units should contribute
comparably, fitting the scaler on the appropriate training or analysis population. `KMeans`
pins `n_clusters`, `init`, `n_init`, `max_iter`, `tol`, and `random_state`. Compare `labels_`,
`cluster_centers_`, `inertia_`, and `n_iter_`, allowing a permutation of cluster names.

Lower inertia always occurs weakly as $k$ grows, so it cannot alone select meaningful $k$.
Inspect multiple initializations, inertia curves, stability under resampling, geometry, and domain
meaning. Do not score clusters against labels that were used to tune the clustering and still call
the procedure unsupervised.

**Checkpoint 6A.** Why must centroid comparisons allow a label permutation?

**Checkpoint 6B.** Why does minimum training inertia select $k=N$ if unconstrained?


In [ ]:
scaled = StandardScaler().fit_transform(X_fit)
km = KMeans(n_clusters=2, init="k-means++", n_init=10, max_iter=100,
            tol=1e-4, random_state=SEED)
km.fit(scaled)
assert km.labels_.shape == (6,)
assert km.cluster_centers_.shape == (2, 2)
assert np.isfinite(km.inertia_)
co_clustering = km.labels_[:, None] == km.labels_[None, :]
assert co_clustering.shape == (6, 6)
assert np.all(np.diag(co_clustering))
print("inertia", km.inertia_, "iterations", km.n_iter_)


## 7. Full classical-model comparison matrix

| Family | Supervision/objective | Geometry & scaling | Output/interpretability | Nonlinear capacity & validation |
|---|---|---|---|---|
| linear regression | supervised MSE | linear; scaling helps regularization | numeric prediction; coefficients | linear; regression CV metrics |
| kNN | supervised local vote | distance-based; scaling crucial | vote/frequency; local examples | nonlinear; tune `k` in CV |
| logistic regression | supervised BCE | linear log-odds; scale for regularization | native probability; coefficients | linear; classification + calibration |
| linear SVM | supervised hinge/margin | linear margin; scaling crucial | score/class; margin normal | linear; tune `C` |
| kernel SVM | supervised hinge/dual | kernel similarity; scaling crucial | score/class; support vectors | nonlinear; tune kernel, `C`, `gamma` |
| decision tree | supervised impurity | axis-aligned; scale-insensitive | leaf frequency/rules | nonlinear; tune depth/pruning |
| forest/bagging | supervised aggregation | tree regions; scale-insensitive | averaged votes; less transparent | nonlinear; CV/OOB plus held-out test |
| boosting | supervised additive correction | base-learner regions | weighted score; sequential ledger | nonlinear; tune rounds/rate/complexity |
| k-means | unsupervised WCSS | centroid/Voronoi; scaling crucial | cluster id/distance; centroids | local optimum; inertia + stability + meaning |

Match task first: labels or no labels, probabilities or decisions, global linearity or nonlinear
structure, interpretability, sample size/prediction cost, and leakage-safe validation. A leaderboard
score without matched splits and metrics is not a model-selection argument.

For a fair benchmark, create one split, then use one seeded `StratifiedKFold` object for every
supervised candidate. Candidate order is also a contract: when mean scores tie exactly, keep the
earlier candidate. `cross_val_score` clones a pipeline inside each fold, so preprocessing fits only
on that fold's training rows. For cluster stability, compare boolean co-clustering matrices
`labels[:, None] == labels[None, :]`; this ignores arbitrary cluster-number permutations.

**Checkpoint 7A.** Which listed families supply native modeled class probabilities?

**Checkpoint 7B.** Which validation evidence is specific to clustering rather than supervised
prediction?


In [ ]:
X_compare, y_compare = make_moons(n_samples=120, noise=0.18, random_state=SEED)
X_compare = X_compare.astype(np.float64)
y_compare = y_compare.astype(np.int64)
X_fit_compare, X_test_compare, y_fit_compare, y_test_compare = train_test_split(
    X_compare, y_compare, test_size=0.25, random_state=SEED, stratify=y_compare
)
folds = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
candidates = [
    make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000,
                                                        random_state=SEED)),
    make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma=1.0)),
    DecisionTreeClassifier(max_depth=3, random_state=SEED),
    RandomForestClassifier(n_estimators=20, max_depth=3, random_state=SEED),
]
fold_scores = np.vstack([
    cross_val_score(candidate, X_fit_compare, y_fit_compare, cv=folds, scoring="accuracy")
    for candidate in candidates
])
assert fold_scores.shape == (4, 4)
print("mean CV scores in declared candidate order", fold_scores.mean(axis=1))


## 8. Pitfalls, exam connections, and closing synthesis

**Pitfalls.** Comparing cluster ids literally across runs, selecting $k$ by training inertia
alone, computing a mean for an empty slice, forgetting squared distances, and silently changing
tie rules all break reproducibility. Across families, preprocessing leakage and mismatched splits
can invalidate an otherwise correct estimator.

**Exam connection.** Expect one Lloyd step in exact arithmetic, a constrained implementation with
shape/dtype/tie policies, or a scenario asking for a model family and defensible validation plan.

**Going deeper.** Later Round 2 work adds new representations and larger models, but the same
comparison discipline remains: objective, geometry, output, assumptions, compute, and validation.

**Checkpoint 8A.** Name three reasons two valid k-means runs may differ.

**Checkpoint 8B.** Give the first two questions to ask before choosing any family in the matrix.


## Checkpoint answers

**1A.** No. **1B.** `inertia_`.

**2A.** `np.argmin` returns the first minimum. **2B.** `(1,3)`.

**3A.** $|C|\|\mu-\bar x_C\|^2$. **3B.** No; different local fixed points can have different
objectives.

**4A.** Poor local minima caused by one initialization. **4B.** The largest-distance row, with a
smaller row-index tie, assigned to the smallest-index empty cluster.

**5A.** Mixing old labels with new centroids can fabricate an increase/decrease unrelated to one
algorithm state. **5B.** Assignment/movement convergence plus `max_iter`.

**6A.** Cluster number is arbitrary. **6B.** Each row can be its own centroid, giving WCSS zero.

**7A.** Logistic regression natively; tree/ensemble leaf or averaged frequencies are available
but require calibration checks, while basic SVM scores are not probabilities. **7B.** Stability
across seeds/resamples and domain/geometry meaning, alongside inertia.

**8A.** Initialization, feature scaling, and empty-cluster/tie policy. **8B.** Is the task
supervised, and what output/metric does the decision require?
